In [ ]:
# --- Complete Evaluation Cell (Original Architecture, NO SE Block) ---
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

print("1. Loading dataset into memory...")
# --- DATA LOADING ---
def sliding_window_sequences(df, window_size=300, step=150):
    feature_cols = df.columns.drop("activityID")
    X, y = [], []
    for start in range(0, len(df) - window_size + 1, step):
        window = df.iloc[start:start + window_size]
        label = window["activityID"].mode()[0]
        X.append(window[feature_cols].values)
        y.append(label)
    return np.array(X, dtype=np.float32), np.array(y)

X_all, y_all = {}, {}
for i in range(1, 9):
    df = pd.read_pickle(rf"/content/CPEN355_FinalProject/preprocessing/data/data_{i}.pkl")
    X, y = sliding_window_sequences(df, window_size=300, step=150)
    X_all[i] = X
    y_all[i] = y

all_labels = np.concatenate(list(y_all.values()))
le = LabelEncoder()
le.fit(all_labels)
for i in range(1, 9):
    y_all[i] = le.transform(y_all[i])


print("2. Building the empty CNN skeleton (Original Architecture)...")
# --- CNN SKELETON (Must exactly match the original .pth files) ---
class PAMAP2_CNN(nn.Module):
    def __init__(self, n_features, n_classes):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv1d(n_features, 64, kernel_size=31, padding=15),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout1d(0.2),

            nn.Conv1d(64, 128, kernel_size=15, padding=7),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout1d(0.2),

            nn.Conv1d(128, 256, kernel_size=7, padding=3),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout1d(0.2),

            nn.Conv1d(256, 256, kernel_size=5, padding=2),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.conv_block(x))


print("3. Evaluating saved .pth models (NO TRAINING!)...")
# --- EVALUATION LOOP ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
results = {}
global_y_true = []
global_y_pred = []

N_FEATURES = X_all[1].shape[2]
N_CLASSES  = len(le.classes_)

for test_subject in range(1, 9):
    # Scale data for this specific fold
    x_train = np.vstack([X_all[i] for i in range(1, 9) if i != test_subject])
    n_train, W, F = x_train.shape
    scaler = StandardScaler()
    scaler.fit(x_train.reshape(-1, F))

    x_test = X_all[test_subject]
    y_true = y_all[test_subject]
    
    x_test_scaled = scaler.transform(x_test.reshape(-1, F)).reshape(x_test.shape[0], W, F)
    x_test_t = torch.tensor(x_test_scaled.transpose(0, 2, 1), dtype=torch.float32)
    test_loader = DataLoader(TensorDataset(x_test_t), batch_size=64)

    # Load the .pth file into the skeleton
    model = PAMAP2_CNN(N_FEATURES, N_CLASSES).to(device)
    model.load_state_dict(torch.load(f'cnn_fold_{test_subject}.pth', map_location=device, weights_only=True))
    model.eval()

    # Make predictions
    all_preds = []
    with torch.no_grad():
        for Xb in test_loader:
            all_preds.extend(model(Xb[0].to(device)).argmax(1).cpu().numpy())
    
    acc = accuracy_score(y_true, all_preds)
    mac_f1 = f1_score(y_true, all_preds, average='macro')
    
    results[test_subject] = {'accuracy': acc, 'macro_f1': mac_f1}
    global_y_true.extend(y_true)
    global_y_pred.extend(all_preds)
    
    print(f"  Subject {test_subject} evaluated. (Acc: {acc:.4f}, F1: {mac_f1:.4f})")


print("\n--- Generating and Saving Final Graphs ---")
# --- PLOTTING ---
subjects = list(results.keys())
accs = [results[s]['accuracy'] for s in subjects]
f1s = [results[s]['macro_f1'] for s in subjects]

# 1. Accuracy Chart
plt.figure(figsize=(8, 6))
plt.bar(subjects, accs, color='steelblue')
plt.axhline(y=np.mean(accs), color='red', linestyle='--', label=f'Mean Acc: {np.mean(accs):.4f}')
plt.xlabel('Subject')
plt.ylabel('Accuracy')
plt.title('LOSO Accuracy per Subject (Saved CNN Models)')
plt.ylim(0, 1.0)
plt.legend()
plt.tight_layout()
plt.savefig('loso_accuracy_saved_cnn.png', dpi=150)
plt.show()

# 2. F1 Score Chart
plt.figure(figsize=(8, 6))
plt.bar(subjects, f1s, color='mediumpurple')
plt.axhline(y=np.mean(f1s), color='red', linestyle='--', label=f'Mean F1: {np.mean(f1s):.4f}')
plt.xlabel('Subject')
plt.ylabel('Macro F1-Score')
plt.title('LOSO Macro F1-Score per Subject (Saved CNN Models)')
plt.ylim(0, 1.0)
plt.legend()
plt.tight_layout()
plt.savefig('loso_f1_saved_cnn.png', dpi=150)
plt.show()

# 3. Global Confusion Matrix
cm = confusion_matrix(global_y_true, global_y_pred)
fig, ax = plt.subplots(figsize=(12, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_.astype(str))
disp.plot(cmap='Blues', ax=ax, xticks_rotation=45) 
plt.title('Global LOSO Confusion Matrix (Saved CNN Models)', fontsize=14)
plt.tight_layout()
plt.savefig('global_confusion_matrix_saved_cnn.png', dpi=150)
plt.show()

ModuleNotFoundError: No module named 'torch'